# The music recommendation systen

## Environment Setup


In [ ]:
## Hey all who are new to this Repo, you'll have to set up a folder in your Google Drive to save data
## The folder should be named "music_reco_project"
## PS: We had to use DEEZER to get audio previews because Spotify doesn't allow direct audio downloads

from google.colab import drive
drive.mount("/content/drive")

import os
PROJECT_DIR = "/content/drive/MyDrive/music_reco_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# persistent folders
DEEZEER_DIR = os.path.join(PROJECT_DIR, "deezer")
PNG_DIR_DRIVE = os.path.join(DEEZEER_DIR, "comp_pngs")
os.makedirs(PNG_DIR_DRIVE, exist_ok=True)

print("Saving checkpoints to:", PROJECT_DIR)


Mounted at /content/drive
Saving checkpoints to: /content/drive/MyDrive/music_reco_project


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import librosa
import math
import requests, time
import os, glob, re
import skimage.io as skio
from skimage.transform import resize
from google.colab import userdata
userdata.get('secretName')

In [4]:
#Load the api client id and secret from file
f = open('apikeys.json')
apikeys = json.load(f)
CLIENT_ID = apikeys['clientId']
CLIENT_SECRET = apikeys['clientSecret']

In [ ]:
def authenticate_token():
    AUTH_URL = 'https://accounts.spotify.com/api/token'

    auth_response = requests.post(AUTH_URL, {
        'grant_type': 'client_credentials',
        'client_id': CLIENT_ID,
        'client_secret': CLIENT_SECRET,
    })
    auth_response_data = auth_response.json()
    access_token = auth_response_data['access_token']
    headers = {
        'Authorization': f'Bearer {access_token}'
    }
    return headers

headers = authenticate_token()

In [ ]:
# base URL of all Spotify API endpoints
BASE_URL = 'https://api.spotify.com/v1/'
genre_seeds = requests.get(BASE_URL + 'recommendations/available-genre-seeds', headers=headers)

## Fetch spotify Artists 


We fetched from the Pop Genre to keep it simple.

In [ ]:
def get_headers():
    return authenticate_token()

def search_artists_by_genre(genre="pop", max_artists=500, page_size=50):
    """
    Returns up to `max_artists` artists tagged with the given genre.
    Note: Spotify's genre tags exist mostly for well-known artists; results aren't exhaustive.
    """
    headers = get_headers()
    seen_ids, artists = set(), []
    offset = 0
    while len(artists) < max_artists:
        params = {
            "q": f'genre:"{genre}"',  # quotes help with multi-word genres
            "type": "artist",
            "limit": page_size,
            "offset": offset,
        }
        r = requests.get(f"{BASE_URL}/search", headers=headers, params=params)
        r.raise_for_status()
        items = r.json().get("artists", {}).get("items", [])
        if not items:
            break
        for a in items:
            if a["id"] not in seen_ids:
                seen_ids.add(a["id"])
                artists.append({"id": a["id"], "name": a["name"]})
                if len(artists) >= max_artists:
                    break
        offset += page_size
        time.sleep(0.1)  # polite pacing vs rate limits
    return artists

pop_artists = search_artists_by_genre("pop", max_artists=50)
print([a["id"] for a in pop_artists[:50]])


['00x1fYSGhdqScXBRpSj3DW', '06HL4z0CvFAxyc27GXpf02', '0RMJOzHDhAKY1o2j0W0vxY', '2dIgFjalVxs4ThymZ67YCE', '1McMsnEElThX1knmY4oliG', '66CXWjxzNUsdJxJ2JdwvnR', '2F3Mdh2idBVOiMTxXoxc10', '6YznhKZUZFVr418x7OUi3z', '4dpARuHxo51G3z768sgnrY', '0YLlTW9rW7ZCy2cA2u3RYk', '45dkTj5sMRSjrmBSBeiHym', '08GQAI4eElDnROBrJRGE0X', '2ylIKKdMukkuprCgY4ZDFE', '53XhwfbYqKCa1cC15pYq2q', '1Xylc3o4UrD53lo9CvFvVg', '6U1dV7aL68N7Gb0Naq34V5', '4gzpq5DPGxSnKTe4SA8HAU', '0LcJLqbBmaGUft1e9Mm8HV', '22wbnEMDvgVIAGdFeek6ET', '7iR5h6yGnTiswjsmj624Rq', '2yNNYQBChuox9A5Ka93BIn', '6qqNVTkY8uBg9cP3Jd7DAH', '77IW5ZK1smDQYYKDCQugXh', '5pKCCKE2ajJHZ9KAiaK11H', '0iVHnv2bQN5iee8J6iCVO4', '1uNFoZAHBGtllmzznpCI3s', '74KM79TiuVKeVCqs8QtB0B', '4G9NDjRyZFDlJKMRL8hx3S', '3D2GUXbtlL3r2d5HJEnsFD', '5oNWzcU0mYK1zDUxBGHIaG', '33qOK5uJ8AR2xuQQAhHump', '6M2wZ9GZgrQXHCFfjv46we', '0Wwji82sLA0Hcvtuak3omb', '7wGBPJk6sHwRCozFfhU09F', '00FQb4jTyendYWaN8pK0wa', '04MtOUkmIDC4LAxDDBjrOY', '3eqjTLE0HfPfh78zjh6TqT', '0fTSzq9jAh4c36UVb4V7CB', '0TnOYISbd1

In [ ]:
pop_artists = search_artists_by_genre("pop", max_artists=200)

pop_df = (
    pd.DataFrame(pop_artists)[["name", "id"]]
      .rename(columns={"name": "artist_name", "id": "artist_id"})
      .drop_duplicates(subset=["artist_id"])
      .reset_index(drop=True)
)

pop_df.head()
# pop_df.to_csv("pop_artists.csv", index=False)  # optional


,artist_name,artist_id
0,Olivia Dean,00x1fYSGhdqScXBRpSj3DW
1,Taylor Swift,06HL4z0CvFAxyc27GXpf02
2,EJAE,0RMJOzHDhAKY1o2j0W0vxY
3,Stray Kids,2dIgFjalVxs4ThymZ67YCE
4,Olivia Rodrigo,1McMsnEElThX1knmY4oliG


In [ ]:
# results =[]
# for idx, genre in enumerate(genre_seeds):
#     params = {
#         'seed_genres':genre,
#         'limit':100
#     }

#     recs = requests.get(BASE_URL + 'recommendations', params=params, headers=headers)
#     rec_tracks = recs.json()['tracks']
#     for track in rec_tracks:
#         artist = track['artists'][0]
#         name = artist['name']
#         id = artist['id']
#         result = {'artist_name':name, 'artist_id':id}
#         results.append(result)
#     print(f'{idx+1} / {len(genre_seeds)}', end='\r')

In [ ]:
# genre_artists_df = pd.DataFrame(results)
genre_artists_df = pop_df
genre_artists_df = genre_artists_df.drop_duplicates().reset_index(drop=True)

In [14]:
genre_artists_df.artist_id

,artist_id
0,00x1fYSGhdqScXBRpSj3DW
1,06HL4z0CvFAxyc27GXpf02
2,0RMJOzHDhAKY1o2j0W0vxY
3,2dIgFjalVxs4ThymZ67YCE
4,1McMsnEElThX1knmY4oliG
...,...
195,6OG9fZ1LKXyL0hShRmmnq1
196,4XQhU3S4TyPkiPIsSu2hmA
197,5l2Xy4aUoJDRSpsYHyOumD
198,7o95ZoZt5ZYn31e9z1Hc0a


In [17]:
genre_artists_df['genres'] = float('nan')
genre_artists_df['popularity'] = float('nan')

We tried to add Genres and then select artists based on those genres, but we decided to just use the Pop genre for simplicity.
Yea and Popularity was just part of the code we took from the internet.

In [18]:
genre_artists_df

,artist_name,artist_id,genres,popularity
0,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,NaN,NaN
1,Taylor Swift,06HL4z0CvFAxyc27GXpf02,NaN,NaN
2,EJAE,0RMJOzHDhAKY1o2j0W0vxY,NaN,NaN
3,Stray Kids,2dIgFjalVxs4ThymZ67YCE,NaN,NaN
4,Olivia Rodrigo,1McMsnEElThX1knmY4oliG,NaN,NaN
...,...,...,...,...
195,Davina Michelle,6OG9fZ1LKXyL0hShRmmnq1,NaN,NaN
196,Diggy Dex,4XQhU3S4TyPkiPIsSu2hmA,NaN,NaN
197,André Hazes Jr.,5l2Xy4aUoJDRSpsYHyOumD,NaN,NaN
198,Natasha Bedingfield,7o95ZoZt5ZYn31e9z1Hc0a,NaN,NaN


In [19]:
genre_artists_full_results = []
for artists in np.array_split(genre_artists_df, 20):
    params = {'ids' : ','.join(list(artists.artist_id))}
    several_artists = requests.get(BASE_URL+'/artists/', params=params, headers=headers)
    for i in artists.index:
        j = i - artists.index[0]
        result = {
            'artist_name': genre_artists_df.loc[i, 'artist_name'],
            'artist_id': genre_artists_df.loc[i, 'artist_id'],
            'genres': several_artists.json()['artists'][j]['genres'],
            'popularity': several_artists.json()['artists'][j]['popularity']
        }
        genre_artists_full_results.append(result)
        print(f'{i+1} / {len(genre_artists_df)}', end= '\r')

genre_artists_df = pd.DataFrame(genre_artists_full_results)

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [20]:
genre_artists_df

,artist_name,artist_id,genres,popularity
0,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],92
1,Taylor Swift,06HL4z0CvFAxyc27GXpf02,[],100
2,EJAE,0RMJOzHDhAKY1o2j0W0vxY,[k-pop],87
3,Stray Kids,2dIgFjalVxs4ThymZ67YCE,"[k-pop, noise music]",87
4,Olivia Rodrigo,1McMsnEElThX1knmY4oliG,[],87
...,...,...,...,...
195,Davina Michelle,6OG9fZ1LKXyL0hShRmmnq1,"[hollands, nederpop]",56
196,Diggy Dex,4XQhU3S4TyPkiPIsSu2hmA,"[nederpop, hollands]",51
197,André Hazes Jr.,5l2Xy4aUoJDRSpsYHyOumD,"[hollands, nederpop]",52
198,Natasha Bedingfield,7o95ZoZt5ZYn31e9z1Hc0a,[],74


## Fetching the tracks 


In [21]:
#extracting tracks from artists
def get_headers():
    return authenticate_token()  # your existing function

def fetch_tracks_for_artists(artists_df, market="US"):
    all_tracks = []
    headers = get_headers()

    for idx, row in artists_df.iterrows():
        artist_id = row["artist_id"]
        artist_name = row["artist_name"]
        artist_genres = row["genres"]
        artist_popularity = row["popularity"]

        url = f"{BASE_URL}/artists/{artist_id}/top-tracks"
        params = {"market": market}

        r = requests.get(url, headers=headers, params=params)

        # basic retry for expired token / rate limit
        while not r.ok:
            if r.status_code == 401:
                headers = get_headers()
                r = requests.get(url, headers=headers, params=params)
            elif r.status_code == 429:
                time.sleep(30)
                r = requests.get(url, headers=headers, params=params)
            else:
                break

        if not r.ok:
            print(f"Skipping {artist_name} ({artist_id}) – status {r.status_code}")
            continue

        for t in r.json().get("tracks", []):
            all_tracks.append({
                "track_id": t["id"],
                "track_name": t["name"],
                "track_preview_link": t.get("preview_url"),
                "track_popularity": t.get("popularity"),
                "track_uri": t.get("uri"),
                "artist_name": artist_name,
                "artist_id": artist_id,
                "artist_genres": artist_genres,
                "artist_popularity": artist_popularity,
                "release_date": t["album"].get("release_date"),
            })

        print(f"{idx+1} / {len(artists_df)} artists processed", end="\r")

    print()
    tracks_raw = pd.DataFrame(all_tracks).drop_duplicates(subset=["track_id"])
    return tracks_raw


tracks_raw = fetch_tracks_for_artists(genre_artists_df)


200 / 200 artists processed


In [22]:
print(tracks_raw.head())

                 track_id                  track_name track_preview_link  \
0  1qbmS6ep2hbBRaEZFpn7BX                  Man I Need               None   
1  6sGIMrtIzQjdzNndVxe397   So Easy (To Fall In Love)               None   
2  7gKxCvTDWwV9wBhdeBbr3l          Nice To Each Other               None   
3  312z6PZ8wwREck8613PkJk            A Couple Minutes               None   
4  3Vd4fHzwS6pBS3muymjiDi  Let Alone The One You Love               None   

   track_popularity                             track_uri  artist_name  \
0                97  spotify:track:1qbmS6ep2hbBRaEZFpn7BX  Olivia Dean   
1                96  spotify:track:6sGIMrtIzQjdzNndVxe397  Olivia Dean   
2                89  spotify:track:7gKxCvTDWwV9wBhdeBbr3l  Olivia Dean   
3                89  spotify:track:312z6PZ8wwREck8613PkJk  Olivia Dean   
4                88  spotify:track:3Vd4fHzwS6pBS3muymjiDi  Olivia Dean   

                artist_id artist_genres  artist_popularity release_date  
0  00x1fYSGhdqScXBRpSj3D

### Filtering the tracks to avoid overflow and maintain relevance

In [24]:
# extracting the tracks that are released on or after 2000

# make sure release_date is string
tracks_raw["release_date"] = tracks_raw["release_date"].astype(str)

# build track_uri in the format spotify:track:<track_id>
tracks_raw["track_uri"] = "spotify:track:" + tracks_raw["track_id"].astype(str)

# extract year and filter
tracks_raw["release_year"] = tracks_raw["release_date"].str[:4].astype(int)
tracks_2000_plus = tracks_raw[tracks_raw["release_year"] >= 2000].copy()

# put columns in exactly the order you want
cols = [
    "track_id",
    "track_name",
    "track_preview_link",
    "track_popularity",
    "track_uri",
    "artist_name",
    "artist_id",
    "artist_genres",
    "artist_popularity",
    "release_date",
]

tracks_2000_plus = tracks_2000_plus[cols].reset_index(drop=True)

tracks_2000_plus.tail()


,track_id,track_name,track_preview_link,track_popularity,track_uri,artist_name,artist_id,artist_genres,artist_popularity,release_date
1628,4iFPsNzNV7V9KJgcOX7TEO,Tu Jaane Na,None,77,spotify:track:4iFPsNzNV7V9KJgcOX7TEO,Pritam,1wRPtKGflJrBx9BmLsSwlU,"[bollywood, hindi pop, desi]",90,2009-11-06
1629,4bD9z9qa4qg9BhryvYWB7c,Kabira,None,78,spotify:track:4bD9z9qa4qg9BhryvYWB7c,Pritam,1wRPtKGflJrBx9BmLsSwlU,"[bollywood, hindi pop, desi]",90,2013-03-30
1630,1UWacd8x8tPPwmrPB1MoBI,Ae Dil Hai Mushkil Title Track,None,75,spotify:track:1UWacd8x8tPPwmrPB1MoBI,Pritam,1wRPtKGflJrBx9BmLsSwlU,"[bollywood, hindi pop, desi]",90,2016-10-26
1631,3P167vmmGRGKHoy7uDugvy,Itni Si Baat Hain,None,67,spotify:track:3P167vmmGRGKHoy7uDugvy,Pritam,1wRPtKGflJrBx9BmLsSwlU,"[bollywood, hindi pop, desi]",90,2016-04-07
1632,1smFN2CLqGROu0J0UyvDfL,Shayad,None,76,spotify:track:1smFN2CLqGROu0J0UyvDfL,Pritam,1wRPtKGflJrBx9BmLsSwlU,"[bollywood, hindi pop, desi]",90,2020-02-14


## Scrapping track preview from DEEZER

In [ ]:
#organised version
# ========= 0) SETUP: Drive + paths =========
from google.colab import drive
drive.mount("/content/drive")


DEEZEER_DIR = "/content/drive/MyDrive/music_reco_project/deezer"  # change if you want
os.makedirs(DEEZEER_DIR, exist_ok=True)

DZ_PATH = os.path.join(DEEZEER_DIR, "dz_df.pkl")

HEADERS = {"User-Agent": "Mozilla/5.0"}

# ========= 1) FILTER OUT CHINESE / NON-LATIN TITLES =========
# Keep only titles containing at least one A-Z letter (drops pure Chinese/Japanese/etc.)
tracks_latin = tracks_2000_plus[
    tracks_2000_plus["track_name"].astype(str).str.contains(r"[A-Za-z]", regex=True)
].reset_index(drop=True)

print("tracks_2000_plus:", len(tracks_2000_plus))
print("tracks_latin (kept for Deezer):", len(tracks_latin))

# ========= 2) CLEAN TITLE + DEEZER SEARCH =========
def clean_title(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\(.*?\)", "", s)                       # remove (...) parts
    s = re.sub(r"-\s*from.*$", "", s, flags=re.I)       # remove "- From ..."
    s = re.sub(r"\s+", " ", s).strip()
    return s

def deezer_search_preview(track_name, artist_name, limit=5):
    """
    Returns (deezer_track_id, deezer_preview_url) or (None, None).
    Uses robust query formats.
    """
    url = "https://api.deezer.com/search"
    queries = [
        f"{artist_name} {track_name}",
        f"{track_name} {artist_name}",
        f'track:"{track_name}" artist:"{artist_name}"',  # fallback
    ]

    for q in queries:
        r = requests.get(url, params={"q": q, "limit": limit}, headers=HEADERS, timeout=20)
        if r.status_code != 200:
            continue

        hits = r.json().get("data", [])
        if not hits:
            continue

        for h in hits:
            if h.get("preview"):
                return h.get("id"), h.get("preview")

    return None, None

# ========= 3) QUICK TEST: Deezer should return something =========
test = requests.get(
    "https://api.deezer.com/search",
    params={"q": "Adele Hello", "limit": 1},
    headers=HEADERS, timeout=20
).json()

print("Deezer test total (Adele Hello):", test.get("total"))

# ========= 4) LOAD SAVED dz_df IF EXISTS, ELSE START NEW =========
if os.path.exists(DZ_PATH):
    dz_df = pd.read_pickle(DZ_PATH)
    print("Loaded dz_df from Drive:", dz_df.shape,
          "| previews:", dz_df["deezer_preview_url"].notna().sum() if "deezer_preview_url" in dz_df.columns else "n/a")
else:
    dz_df = tracks_latin.copy()
    dz_df["deezer_track_id"] = None
    dz_df["deezer_preview_url"] = None
    print("Starting new dz_df:", dz_df.shape)

# IMPORTANT: ensure dz_df is based on tracks_latin (in case old saved file was from another dataset)
# We'll align by track_id and keep the current tracks_latin rows.
dz_df = tracks_latin.merge(
    dz_df[["track_id", "deezer_track_id", "deezer_preview_url"]] if "deezer_preview_url" in dz_df.columns else tracks_latin[["track_id"]],
    on="track_id",
    how="left"
)

# If columns missing after merge, create them
if "deezer_track_id" not in dz_df.columns:
    dz_df["deezer_track_id"] = None
if "deezer_preview_url" not in dz_df.columns:
    dz_df["deezer_preview_url"] = None

print("dz_df aligned shape:", dz_df.shape)

# ========= 5) FILL ONLY MISSING PREVIEWS + SAVE PROGRESS =========
SAVE_EVERY = 50
SLEEP = 0.05

missing_idx = dz_df[dz_df["deezer_preview_url"].isna()].index.tolist()
print("Rows missing preview to query:", len(missing_idx))

for n, i in enumerate(missing_idx, start=1):
    title = clean_title(dz_df.at[i, "track_name"])
    artist = dz_df.at[i, "artist_name"]

    dz_id, dz_preview = deezer_search_preview(title, artist)
    dz_df.at[i, "deezer_track_id"] = dz_id
    dz_df.at[i, "deezer_preview_url"] = dz_preview

    if n % 25 == 0:
        found = dz_df["deezer_preview_url"].notna().sum()
        print(f"Queried {n}/{len(missing_idx)} | previews found: {found}", end="\r")

    if n % SAVE_EVERY == 0:
        dz_df.to_pickle(DZ_PATH)

    time.sleep(SLEEP)

# final save
dz_df.to_pickle(DZ_PATH)
print("\nSaved dz_df to:", DZ_PATH)

# ========= 6) FINAL FILTER: keep only rows with previews =========
dz_df = dz_df[dz_df["deezer_preview_url"].notna()].reset_index(drop=True)
print("Tracks with Deezer previews:", len(dz_df), "/", len(tracks_latin))
dz_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
tracks_2000_plus: 1633
tracks_latin (kept for Deezer): 1626
Deezer test total (Adele Hello): 232
Loaded dz_df from Drive: (1163, 12) | previews: 0
dz_df aligned shape: (1626, 12)
Rows missing preview to query: 1626

Saved dz_df to: /content/drive/MyDrive/music_reco_project/deezer/dz_df.pkl
Tracks with Deezer previews: 1619 / 1626


,track_id,track_name,track_preview_link,track_popularity,track_uri,artist_name,artist_id,artist_genres,artist_popularity,release_date,deezer_track_id,deezer_preview_url
0,1qbmS6ep2hbBRaEZFpn7BX,Man I Need,None,97,spotify:track:1qbmS6ep2hbBRaEZFpn7BX,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],92,2025-08-15,3503857201,https://cdnt-preview.dzcdn.net/api/1/1/0/b/9/0...
1,6sGIMrtIzQjdzNndVxe397,So Easy (To Fall In Love),None,96,spotify:track:6sGIMrtIzQjdzNndVxe397,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],92,2025-09-26,3562948071,https://cdnt-preview.dzcdn.net/api/1/1/8/a/d/0...
2,7gKxCvTDWwV9wBhdeBbr3l,Nice To Each Other,None,89,spotify:track:7gKxCvTDWwV9wBhdeBbr3l,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],92,2025-05-30,3562948041,https://cdnt-preview.dzcdn.net/api/1/1/8/d/1/0...
3,312z6PZ8wwREck8613PkJk,A Couple Minutes,None,89,spotify:track:312z6PZ8wwREck8613PkJk,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],92,2025-09-26,3562948141,https://cdnt-preview.dzcdn.net/api/1/1/2/0/e/0...
4,3Vd4fHzwS6pBS3muymjiDi,Let Alone The One You Love,None,88,spotify:track:3Vd4fHzwS6pBS3muymjiDi,Olivia Dean,00x1fYSGhdqScXBRpSj3DW,[pop soul],92,2025-09-26,3562948081,https://cdnt-preview.dzcdn.net/api/1/1/0/7/b/0...


In [32]:
dz_df.to_csv("/content/drive/MyDrive/music_reco_project/deezer/dz_df.csv", index=False)
print("Saved dz_df.csv to Drive")


Saved dz_df.csv to Drive


In [ ]:
track_id = int(dz_df.loc[0, "deezer_track_id"])
r = requests.get(f"https://api.deezer.com/track/{track_id}")
print(r.status_code)
print(r.json().get("preview"))


200
https://cdnt-preview.dzcdn.net/api/1/1/0/b/9/0/0b9ca2e62ea50bd7c97faa79b5b483f2.mp3?hdnea=exp=1767867285~acl=/api/1/1/0/b/9/0/0b9ca2e62ea50bd7c97faa79b5b483f2.mp3*~data=user_id=0,application_id=42~hmac=f97386f77d8e31ab8f5b40bfc6cdb4190820a195ffbdade5b796dfe0bac223ac


## Converting Tracks to Mel Spectrograms

In [ ]:
# ===================== STEP C: Download Deezer previews + create composite PNGs =====================


# ----- Paths (Drive) -----
DEEZEER_DIR = "/content/drive/MyDrive/music_reco_project/deezer"
DZ_PATH = os.path.join(DEEZEER_DIR, "dz_df.pkl")
PNG_DIR_DRIVE = os.path.join(DEEZEER_DIR, "comp_pngs")
TMP_MP3_DIR = "/content/tmp_deezer_mp3s"

os.makedirs(PNG_DIR_DRIVE, exist_ok=True)
os.makedirs(TMP_MP3_DIR, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0"}

# ----- Load -----
dz_df = pd.read_pickle(DZ_PATH)
dz_df = dz_df[dz_df["deezer_track_id"].notna()].copy().reset_index(drop=True)
dz_df["track_id"] = dz_df["track_id"].astype(str)
dz_df["deezer_track_id"] = dz_df["deezer_track_id"].astype(int)

print("Loaded dz_df:", dz_df.shape)

def png_path(track_id: str) -> str:
    return os.path.join(PNG_DIR_DRIVE, f"{track_id}.png")

def convert_audio_to_composite_image(filepath_to_audio, filepath_to_save,
                                     image_size=(128, 512), n_mels=128, fmax=8000):
    signal, sr = librosa.load(filepath_to_audio, sr=None)

    mels = librosa.power_to_db(
        librosa.feature.melspectrogram(y=signal, sr=sr, n_mels=n_mels, fmax=fmax),
        ref=np.max
    )
    mel_img = (((80 + mels) / 80) * 255)
    mel_img = np.flip(mel_img, axis=0)
    mel_img = resize(mel_img, image_size, preserve_range=True).astype(np.uint8)

    # These 2 are just for testing, we can remove them later

    mfcc = librosa.power_to_db(
        librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=128),
        ref=np.max
    )
    mfcc_img = (((80 + mfcc) / 80) * 255)
    mfcc_img = np.flip(mfcc_img, axis=0)
    mfcc_img = resize(mfcc_img, image_size, preserve_range=True).astype(np.uint8)

    chroma = librosa.feature.chroma_cqt(y=signal, sr=sr)
    chroma_img = resize(chroma * 255, image_size, preserve_range=True).astype(np.uint8)

    composite = np.dstack((mel_img, mfcc_img, chroma_img))
    skio.imsave(filepath_to_save, composite)

def get_preview_from_track_endpoint(deezer_track_id: int):
    """Fetch a fresh preview URL from Deezer API."""
    r = requests.get(f"https://api.deezer.com/track/{deezer_track_id}", headers=HEADERS, timeout=20)
    if r.status_code != 200:
        return None
    return r.json().get("preview")

def download_preview(url: str, out_path: str):
    r = requests.get(url, headers=HEADERS, timeout=30, allow_redirects=True)
    if not r.ok:
        return False
    ctype = (r.headers.get("Content-Type") or "").lower()
    if "audio" not in ctype and "mpeg" not in ctype:
        return False
    if not r.content or len(r.content) < 50_000:
        return False
    with open(out_path, "wb") as f:
        f.write(r.content)
    return True

# ----- Resume-safe: only process missing PNGs -----
todo = []
for i, row in dz_df.iterrows():
    if not os.path.exists(png_path(row["track_id"])):
        todo.append(i)

print("Already have PNGs:", len(dz_df) - len(todo))
print("To generate:", len(todo))

success = 0
fails = 0

for n, i in enumerate(todo, start=1):
    track_id = dz_df.at[i, "track_id"]
    deezer_tid = int(dz_df.at[i, "deezer_track_id"])
    out_png = png_path(track_id)
    tmp_mp3 = os.path.join(TMP_MP3_DIR, f"{track_id}.mp3")

    # 1) get fresh preview url
    preview_url = get_preview_from_track_endpoint(deezer_tid)
    if not preview_url:
        fails += 1
        continue

    # 2) download mp3
    ok = download_preview(preview_url, tmp_mp3)
    if not ok:
        fails += 1
        continue

    # 3) convert to image
    try:
        convert_audio_to_composite_image(tmp_mp3, out_png)
        success += 1
    except Exception:
        fails += 1
    finally:
        if os.path.exists(tmp_mp3):
            os.remove(tmp_mp3)

    if n % 25 == 0 or n == len(todo):
        print(f"Processed {n}/{len(todo)} | new PNGs: {success} | fails: {fails}", flush=True)

    time.sleep(0.05)

print("\nDone.")
print("Total PNGs now in Drive:", len(glob.glob(os.path.join(PNG_DIR_DRIVE, '*.png'))))


Loaded dz_df: (1619, 12)
Already have PNGs: 0
To generate: 1619
Processed 25/1619 | new PNGs: 25 | fails: 0
Processed 50/1619 | new PNGs: 50 | fails: 0
Processed 75/1619 | new PNGs: 75 | fails: 0
Processed 100/1619 | new PNGs: 100 | fails: 0
Processed 125/1619 | new PNGs: 125 | fails: 0
Processed 150/1619 | new PNGs: 150 | fails: 0
Processed 175/1619 | new PNGs: 175 | fails: 0
Processed 200/1619 | new PNGs: 200 | fails: 0
Processed 225/1619 | new PNGs: 225 | fails: 0
Processed 250/1619 | new PNGs: 250 | fails: 0
Processed 275/1619 | new PNGs: 275 | fails: 0
Processed 300/1619 | new PNGs: 300 | fails: 0
Processed 325/1619 | new PNGs: 325 | fails: 0
Processed 350/1619 | new PNGs: 350 | fails: 0
Processed 375/1619 | new PNGs: 375 | fails: 0
Processed 400/1619 | new PNGs: 400 | fails: 0
Processed 425/1619 | new PNGs: 425 | fails: 0
Processed 450/1619 | new PNGs: 450 | fails: 0
Processed 475/1619 | new PNGs: 475 | fails: 0
Processed 500/1619 | new PNGs: 500 | fails: 0
Processed 525/1619 | n

In [ ]:

PNG_DIR_DRIVE = "/content/drive/MyDrive/music_reco_project/deezer/comp_pngs"
print("PNGs:", len(glob.glob(os.path.join(PNG_DIR_DRIVE, "*.png"))))


PNGs: 1619


In [ ]:
from PIL import Image

DEEZEER_DIR = "/content/drive/MyDrive/music_reco_project/deezer"
DZ_PATH = os.path.join(DEEZEER_DIR, "dz_df.pkl")
PNG_DIR_DRIVE = os.path.join(DEEZEER_DIR, "comp_pngs")

# --- controls ---
TARGET_H, TARGET_W = 128, 256     # smaller than 128x512
MAX_TRAIN_IMAGES = 600            # reduce training size (adjust: 300, 600, 1000...)
RANDOM_STATE = 42

dz_df = pd.read_pickle(DZ_PATH)
dz_df = dz_df[dz_df["deezer_track_id"].notna()].copy()
dz_df["track_id"] = dz_df["track_id"].astype(str)

png_paths = glob.glob(os.path.join(PNG_DIR_DRIVE, "*.png"))
print("Found PNGs:", len(png_paths))

id_to_path = {os.path.splitext(os.path.basename(p))[0]: p for p in png_paths}

dz_df_with_img = dz_df[dz_df["track_id"].isin(id_to_path.keys())].copy()
dz_df_with_img["png_path"] = dz_df_with_img["track_id"].map(id_to_path)

print("Rows with images:", len(dz_df_with_img))

# --- take a smaller subset for faster training ---
if len(dz_df_with_img) > MAX_TRAIN_IMAGES:
    dz_df_with_img = dz_df_with_img.sample(n=MAX_TRAIN_IMAGES, random_state=RANDOM_STATE).reset_index(drop=True)
    print("Subsampled to:", len(dz_df_with_img))

def load_image_resized(path):
    img = Image.open(path).convert("RGB")
    # resize to smaller size (W, H) for PIL
    img = img.resize((TARGET_W, TARGET_H))
    return np.array(img, dtype=np.float32) / 255.0

X = np.stack([load_image_resized(p) for p in dz_df_with_img["png_path"].tolist()], axis=0)
print("X shape:", X.shape)  # (N, 128, 256, 3)



Found PNGs: 1619
Rows with images: 1619
Subsampled to: 600
X shape: (600, 128, 256, 3)


## Training our own Autoencoder

In [41]:
#train autoencoder
from sklearn.model_selection import train_test_split

X_train, X_val = train_test_split(X, test_size=0.1, random_state=42)
print(X_train.shape, X_val.shape)


(540, 128, 256, 3) (60, 128, 256, 3)


In [ ]:
!pip -q install tensorflow
import tensorflow as tf
from tensorflow.keras import layers, Model

LATENT_DIM = 64
inp = layers.Input(shape=(128, 256, 3))

# Encoder
x = layers.Conv2D(32, 3, activation="relu", padding="same")(inp)
x = layers.MaxPooling2D(2, padding="same")(x)   # 64x128
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)   # 32x64
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.MaxPooling2D(2, padding="same")(x)   # 16x32

x = layers.Flatten()(x)
z = layers.Dense(LATENT_DIM, name="latent")(x)

# Decoder
x = layers.Dense(16 * 32 * 128, activation="relu")(z)
x = layers.Reshape((16, 32, 128))(x)

x = layers.UpSampling2D(2)(x)                   # 32x64
x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
x = layers.UpSampling2D(2)(x)                   # 64x128
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.UpSampling2D(2)(x)                   # 128x256
x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)

out = layers.Conv2D(3, 3, activation="sigmoid", padding="same")(x)

autoencoder = Model(inp, out)
encoder = Model(inp, z)

autoencoder.compile(optimizer="adam", loss="mse")

history = autoencoder.fit(
    X_train, X_train,
    validation_data=(X_val, X_val),
    epochs=10,
    batch_size=32,
    shuffle=True
)


Epoch 1/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 261s 14s/step - loss: 0.0569 - val_loss: 0.0478
Epoch 2/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 232s 14s/step - loss: 0.0476 - val_loss: 0.0458
Epoch 3/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 283s 15s/step - loss: 0.0461 - val_loss: 0.0445
Epoch 4/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 243s 14s/step - loss: 0.0441 - val_loss: 0.0416
Epoch 5/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 254s 14s/step - loss: 0.0420 - val_loss: 0.0412
Epoch 6/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 248s 15s/step - loss: 0.0410 - val_loss: 0.0405
Epoch 7/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 234s 14s/step - loss: 0.0406 - val_loss: 0.0401
Epoch 8/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 264s 14s/step - loss: 0.0401 - val_loss: 0.0394
Epoch 9/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 233s 14s/step - loss: 0.0392 - val_loss: 0.0388
Epoch 10/10
17/17 ━━━━━━━━━━━━━━━━━━━━ 233s 14s/step - loss: 0.0385 - val_loss: 0.0376


In [43]:
Z = encoder.predict(X, batch_size=64)
print("Embeddings shape:", Z.shape)  # (N, LATENT_DIM)


10/10 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step
Embeddings shape: (600, 64)


## Cosine comparison of Latent Representations (from encoder)

In [44]:
import numpy as np

# normalize for cosine
Z_norm = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

def top_k_similar(idx, k=10):
    sims = Z_norm @ Z_norm[idx]          # cosine similarity with all songs
    best = np.argsort(-sims)[:k+1]       # include itself
    best = best[best != idx][:k]         # remove itself
    return best, sims[best]

q = 0
neighbors, scores = top_k_similar(q, k=10)
dz_df_with_img.loc[neighbors, ["track_name", "artist_name"]].assign(score=scores)


,track_name,artist_name,score
374,Adio Amore Adio,Jannes,0.982849
329,Dónde Están Corazón,Enrique Iglesias,0.973585
468,Nu Wij Niet Meer Praten,Jaap Reesema,0.973089
542,Voor Je ‘t Weet,Tino Martin,0.970928
487,Slow It Down,Benson Boone,0.963155
237,Shameless,Camila Cabello,0.955937
98,Samen Voor Altijd,Marco Borsato,0.955433
339,"17 Miljoen Mensen - Live@538 In Ahoy, Rotterda...",Davina Michelle,0.954708
399,Against All Odds (Take a Look at Me Now) - 201...,Phil Collins,0.952065
260,vampire,Olivia Rodrigo,0.951848


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assume you already have:
# - autoencoder  (model: inp -> out)
# - encoder      (model: inp -> z)

# Pick an index to visualize
idx = 0
x = X_val[idx:idx+1]          # or X_train

z = encoder.predict(x)
x_rec = autoencoder.predict(x)

x_img = x[0]
x_rec_img = x_rec[0]
z_vec = z[0]

latent_side = int(np.sqrt(z_vec.shape[0]))
z_img = z_vec.reshape(latent_side, latent_side)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Input image
axes[0].imshow(x_img)
axes[0].set_title("Input")
axes[0].axis("off")

# Latent "image"
im1 = axes[1].imshow(z_img, cmap="viridis")
axes[1].set_title("Latent (heatmap)")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# Reconstructed image
axes[2].imshow(x_rec_img)
axes[2].set_title("Reconstruction")
axes[2].axis("off")

plt.tight_layout()
plt.show()


## Recommendations

In [45]:
trackid_to_idx = {tid: i for i, tid in enumerate(dz_df_with_img["track_id"].astype(str).tolist())}

def recommend_by_track_id(track_id, k=10):
    idx = trackid_to_idx[str(track_id)]
    neighbors, scores = top_k_similar(idx, k=k)
    out = dz_df_with_img.loc[neighbors, ["track_id","track_name","artist_name"]].copy()
    out["cosine_score"] = scores
    return out

recommend_by_track_id(dz_df_with_img.loc[0,"track_id"], k=10)


,track_id,track_name,artist_name,cosine_score
374,53AddGhMgfIE85Az2Ipovu,Adio Amore Adio,Jannes,0.982849
329,3bsem3DOS0VuSB8OPGAdiU,Dónde Están Corazón,Enrique Iglesias,0.973585
468,7d89CDPikSfHfQjw0WYnxB,Nu Wij Niet Meer Praten,Jaap Reesema,0.973089
542,0kQW438RAa27ixstP4bIhD,Voor Je ‘t Weet,Tino Martin,0.970928
487,51eSHglvG1RJXtL3qI5trr,Slow It Down,Benson Boone,0.963155
237,2ogKhhoMClkFXek7ZgxAhN,Shameless,Camila Cabello,0.955937
98,3zJ2U5rN8Pl1Aqnxar7TOw,Samen Voor Altijd,Marco Borsato,0.955433
339,5Kgr8p9CdaX4on2nSbr2Lz,"17 Miljoen Mensen - Live@538 In Ahoy, Rotterda...",Davina Michelle,0.954708
399,63CHa6rmamv9OsehkRD8oz,Against All Odds (Take a Look at Me Now) - 201...,Phil Collins,0.952065
260,1kuGVB7EU95pJObxwvfwKS,vampire,Olivia Rodrigo,0.951848


## Widgets to Display Recommendations - UI

In [54]:
#drop down to choose artists
!pip -q install ipywidgets
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

dz_df_with_img = dz_df_with_img.reset_index(drop=True)

# Normalize embeddings for cosine similarity
Z_norm = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

def top_k_similar(idx, k=10):
    sims = Z_norm @ Z_norm[idx]
    best = np.argsort(-sims)[:k+1]
    best = best[best != idx][:k]
    return best, sims[best]

# --- Build dropdown lists ---
artists = sorted(dz_df_with_img["artist_name"].astype(str).unique().tolist())

artist_dd = widgets.Dropdown(
    options=artists,
    description="Artist:",
    layout=widgets.Layout(width="30%")
)

song_dd = widgets.Dropdown(
    options=[],
    description="Song:",
    layout=widgets.Layout(width="30%")
)

k_slider = widgets.IntSlider(description="Top K:", value=10, min=3, max=20)
btn = widgets.Button(description="Recommend", button_style="success")
out = widgets.Output()

def update_songs(*args):
    chosen_artist = artist_dd.value
    songs = dz_df_with_img.loc[
        dz_df_with_img["artist_name"].astype(str) == str(chosen_artist),
        "track_name"
    ].astype(str).unique().tolist()
    songs = sorted(songs)
    song_dd.options = songs
    if songs:
        song_dd.value = songs[0]

artist_dd.observe(update_songs, names="value")
update_songs()  # initialize song list

def on_click(_):
    with out:
        clear_output()
        a = str(artist_dd.value)
        s = str(song_dd.value)

        # find the row index for this exact artist+song
        matches = dz_df_with_img[
            (dz_df_with_img["artist_name"].astype(str) == a) &
            (dz_df_with_img["track_name"].astype(str) == s)
        ]
        if matches.empty:
            print("Could not find this song in the dataset (unexpected).")
            return

        idx = int(matches.index[0])

        print("Query song:")
        display(dz_df_with_img.loc[[idx], ["track_name","artist_name","track_id"]])

        neighbors, scores = top_k_similar(idx, k=k_slider.value)
        recs = dz_df_with_img.loc[neighbors, ["track_name","artist_name","track_id"]].copy()
        recs["cosine_score"] = scores
        display(recs.reset_index(drop=True))

btn.on_click(on_click)

display(artist_dd, song_dd, k_slider, btn, out)


Dropdown(description='Artist:', layout=Layout(width='30%'), options=('5 Seconds of Summer', 'Acda en de Munnik…

Dropdown(description='Song:', layout=Layout(width='30%'), options=('Everyone’s A Star!', "I'm Scared I’ll Neve…

IntSlider(value=10, description='Top K:', max=20, min=3)

Button(button_style='success', description='Recommend', style=ButtonStyle())

Output()

In [51]:
#to find the lowest cosine similar score
import numpy as np

# build mapping track_id -> index (if you don't already have it)
trackid_to_idx = {tid: i for i, tid in enumerate(dz_df_with_img["track_id"].astype(str).tolist())}

query_track_id = "53iuhJlwXhSER5J2IYYv1W"
q_idx = trackid_to_idx[query_track_id]

# cosine similarities to all songs
sims = Z_norm @ Z_norm[q_idx]

# exclude itself
sims[q_idx] = np.inf

# find lowest similarity
worst_idx = int(np.argmin(sims))
worst_score = float(sims[worst_idx])

print("Query:")
print(dz_df_with_img.loc[q_idx, ["track_name","artist_name","track_id"]])
print("\nLeast similar:")
print(dz_df_with_img.loc[worst_idx, ["track_name","artist_name","track_id"]])
print("\nLowest cosine similarity:", worst_score)


Query:
track_name        The Fate of Ophelia
artist_name              Taylor Swift
track_id       53iuhJlwXhSER5J2IYYv1W
Name: 553, dtype: object

Least similar:
track_name                    Haunted
artist_name                   Beyoncé
track_id       7cioKB5CHVzk09SOtTyn0T
Name: 278, dtype: object

Lowest cosine similarity: -0.14886973798274994


## Some overall results for documentation

In [53]:
# compare the numbers between the tracks
import numpy as np
import pandas as pd

# --- mapping track_id -> row index ---
trackid_to_idx = {tid: i for i, tid in enumerate(dz_df_with_img["track_id"].astype(str).tolist())}

query_track_id = "53iuhJlwXhSER5J2IYYv1W"
q = trackid_to_idx[query_track_id]

# --- cosine similarities against all songs ---
sims = Z_norm @ Z_norm[q]
sims[q] = -np.inf  # exclude itself for max

best = int(np.argmax(sims))       # highest cosine
best_score = float(sims[best])

sims2 = Z_norm @ Z_norm[q]
sims2[q] = np.inf                 # exclude itself for min
worst = int(np.argmin(sims2))     # lowest cosine
worst_score = float(sims2[worst])

print("Query:", dz_df_with_img.loc[q, ["track_name","artist_name","track_id"]].to_dict())
print("Most similar:", dz_df_with_img.loc[best, ["track_name","artist_name","track_id"]].to_dict(), "score=", best_score)
print("Least similar:", dz_df_with_img.loc[worst, ["track_name","artist_name","track_id"]].to_dict(), "score=", worst_score)

# --- compare embeddings (use raw Z, not normalized, for differences) ---
z_q = Z[q]
z_best = Z[best]
z_worst = Z[worst]

diff_best = z_best - z_q
diff_worst = z_worst - z_q

# summary numbers
print("\nVector difference summary (L2 norm):")
print("Query vs Most similar:", float(np.linalg.norm(diff_best)))
print("Query vs Least similar:", float(np.linalg.norm(diff_worst)))

print("\nMean abs difference per dimension:")
print("Query vs Most similar:", float(np.mean(np.abs(diff_best))))
print("Query vs Least similar:", float(np.mean(np.abs(diff_worst))))

# show top dimensions that differ the most
TOP_D = 10
top_dims_best = np.argsort(-np.abs(diff_best))[:TOP_D]
top_dims_worst = np.argsort(-np.abs(diff_worst))[:TOP_D]

best_table = pd.DataFrame({
    "dim": top_dims_best,
    "query": z_q[top_dims_best],
    "most_similar": z_best[top_dims_best],
    "delta": diff_best[top_dims_best],
    "abs_delta": np.abs(diff_best[top_dims_best]),
}).sort_values("abs_delta", ascending=False)

worst_table = pd.DataFrame({
    "dim": top_dims_worst,
    "query": z_q[top_dims_worst],
    "least_similar": z_worst[top_dims_worst],
    "delta": diff_worst[top_dims_worst],
    "abs_delta": np.abs(diff_worst[top_dims_worst]),
}).sort_values("abs_delta", ascending=False)

print("\nTop differing dimensions: Query vs Most similar")
display(best_table)

print("\nTop differing dimensions: Query vs Least similar")
display(worst_table)



Query: {'track_name': 'The Fate of Ophelia', 'artist_name': 'Taylor Swift', 'track_id': '53iuhJlwXhSER5J2IYYv1W'}
Most similar: {'track_name': 'ni pedo', 'artist_name': 'Peso Pluma', 'track_id': '7xLfNI4gRNTu6FyXJlzBW5'} score= 0.9919291138648987
Least similar: {'track_name': 'Haunted', 'artist_name': 'Beyoncé', 'track_id': '7cioKB5CHVzk09SOtTyn0T'} score= -0.14886973798274994

Vector difference summary (L2 norm):
Query vs Most similar: 1.5819363594055176
Query vs Least similar: 26.794342041015625

Mean abs difference per dimension:
Query vs Most similar: 0.15997666120529175
Query vs Least similar: 2.635936737060547

Top differing dimensions: Query vs Most similar


,dim,query,most_similar,delta,abs_delta
0,42,-2.067090,-1.609162,0.457928,0.457928
1,63,1.355580,1.805454,0.449875,0.449875
2,15,-0.605121,-0.980933,-0.375812,0.375812
3,47,-3.087553,-2.726750,0.360803,0.360803
4,38,1.094890,1.416149,0.321259,0.321259
5,7,-0.196270,0.114474,0.310745,0.310745
6,27,-0.706552,-1.015221,-0.308670,0.308670
7,61,1.106853,1.404057,0.297204,0.297204
8,5,-0.473601,-0.185099,0.288502,0.288502
9,53,-1.171100,-1.445074,-0.273974,0.273974



Top differing dimensions: Query vs Least similar


,dim,query,least_similar,delta,abs_delta
0,46,-1.769361,5.868389,7.637750,7.637750
1,21,2.013977,-5.355186,-7.369163,7.369163
2,16,1.836614,-5.478482,-7.315096,7.315096
3,23,2.230580,-4.995529,-7.226109,7.226109
4,42,-2.067090,4.746999,6.814089,6.814089
5,62,-2.399281,3.820706,6.219986,6.219986
6,14,-1.461472,4.755936,6.217408,6.217408
7,47,-3.087553,2.844587,5.932140,5.932140
8,28,1.794546,-3.143138,-4.937684,4.937684
9,9,0.977377,-3.898306,-4.875683,4.875683
